# Name That LLM — multi-view n-gram LinearSVC + public extra data

Four TF-IDF views (char 3-5, word 1-3, word-shape 2-6, function-word 1-3) feeding a class-balanced LinearSVC.
Training data = competition train + [prvsiyan's Extended Name That LLM Dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset) (`passages_clean.csv`, CC BY-SA 4.0).

CV (GroupKFold by prompt, scored on competition passages only): host only 0.385, + public data 0.392.

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

def find(pattern, local):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    return hits[0] if hits else local

train = pd.read_csv(find('train.csv', '../data/train.csv'))
test = pd.read_csv(find('test.csv', '../data/test.csv'))
test['text'] = test['text'].fillna('').astype(str)
extra = pd.read_csv(find('passages_clean.csv', '../data/external/prvsiyan/passages_clean.csv'))[['model', 'text']]
extra['text'] = extra.text.astype(str).str.replace(r'\s*\n+\s*', ' ', regex=True).str.strip()
fit = pd.concat([train[['model', 'text']], extra], ignore_index=True)
print('train', train.shape, 'extra', extra.shape, 'test', test.shape)

In [ ]:
FUNCTION_WORDS = set('''a an the and or but if while although because as so than that which who whom whose this these
those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from
with without for of by over under between through into about is are was were be been being have has had do does did can
could will would shall should may might must not no nor all any some each every both either neither other another such
only also just very too more most less least here there when where why how what whether then now still yet even much many
few little'''.split())
# (view, ngram lo, ngram hi, max_features, weight)
VIEWS = [('char', 3, 5, 250000, 1.0), ('word', 1, 3, 180000, 0.7), ('shape', 2, 6, 60000, 0.5),
         ('function', 1, 3, 50000, 0.5)]

def shape_text(t):  # letters -> x / X
    return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', t)

def function_text(t):  # content words -> CONTENT
    return ' '.join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else 'CONTENT'
                    for w in re.findall(r'\w+|[^\w\s]', t))

A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == 'shape' else function_text if view == 'function' else (lambda s: s)
    analyzer = 'char' if view == 'shape' else 'word' if view == 'function' else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == 'word':
        opts['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit.text]) * weight)
    B.append(vec.transform([prep(t) for t in test.text]) * weight)

svc = LinearSVC(C=0.5, max_iter=5000, class_weight='balanced', random_state=42)
svc.fit(hstack(A, format='csr'), fit.model)
pred = svc.predict(hstack(B, format='csr'))

In [ ]:
VALID = set(train['model'].unique())
assert len(VALID) == 18

submission = pd.DataFrame({'id': test['id'], 'model': pred})
assert submission['id'].is_unique and submission['model'].isin(VALID).all() and len(submission) == len(test)
submission.to_parquet('submission.parquet', index=False)
submission.head()